# Classificação Supervisionada com Agglomerative Clustering no Dataset Adult

Este notebook apresenta um experimento didático de classificação supervisionada utilizando o algoritmo de Agglomerative Clustering no conjunto de dados Adult. O fluxo segue uma estrutura passo a passo, incluindo carregamento, pré-processamento, divisão dos dados, análise do número de clusters (método do cotovelo), implementação, treinamento, avaliação e análise dos resultados.

Cada etapa é explicada em detalhes para facilitar o entendimento e a reprodutibilidade do experimento.

In [ ]:
# Importação das bibliotecas necessárias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

# Configuração de estilo dos gráficos
sns.set(style="whitegrid")

## 1. Carregamento do Dataset

Nesta etapa, o conjunto de dados Adult será carregado. O dataset contém informações demográficas e de renda, sendo amplamente utilizado em tarefas de classificação. Vamos explorar rapidamente suas dimensões e amostras.

In [ ]:
# Carregamento do dataset Adult
# O arquivo deve estar em 'src/data/adult.csv' ou caminho equivalente

dataset_path = '../data/adult.csv'  # ajuste o caminho se necessário

df = pd.read_csv(dataset_path)
print(f"Formato do dataset: {df.shape}")
df.head()

## 2. Pré-processamento dos Dados

Nesta etapa, serão realizados tratamentos como remoção de valores ausentes, codificação de variáveis categóricas e normalização dos dados. O objetivo é preparar o dataset para o uso em algoritmos de clustering.

In [ ]:
# Pré-processamento dos dados
# 1. Remover valores ausentes
# 2. Codificar variáveis categóricas
# 3. Normalizar os dados

df = df.replace('?', np.nan)
df = df.dropna()

# Separar features e target
target_col = 'income'  # ajuste se necessário
X = df.drop(target_col, axis=1)
y = df[target_col]

# Codificação de variáveis categóricas
def encode_features(X):
    X_encoded = X.copy()
    for col in X_encoded.select_dtypes(include=['object']).columns:
        X_encoded[col] = LabelEncoder().fit_transform(X_encoded[col])
    return X_encoded

X_encoded = encode_features(X)

# Normalização
def normalize_features(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X)

X_norm = normalize_features(X_encoded)

# Exibir shape final
display(pd.DataFrame(X_norm).head())

## 3. Divisão em Conjuntos de Treino e Teste

Agora, vamos dividir os dados em conjuntos de treino e teste para avaliar o desempenho do modelo de clustering supervisionado.

In [ ]:
# Divisão dos dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.3, random_state=42, stratify=y)

print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")

## 4. Escolha do Número de Clusters (Método do Cotovelo)

Para determinar o número ideal de clusters, utilizaremos o método do cotovelo, que avalia a inércia (dentro dos clusters) para diferentes valores de k.

In [ ]:
# Método do cotovelo para Agglomerative Clustering
# Usando linkage para visualizar o dendrograma

plt.figure(figsize=(12, 5))
linked = linkage(X_train, method='ward')
dendrogram(linked, truncate_mode='level', p=10)
plt.title('Dendrograma - Método do Cotovelo')
plt.xlabel('Amostras')
plt.ylabel('Distância')
plt.show()

# Sugestão: escolha visual do número de clusters (exemplo: 2)

## 5. Implementação e Treinamento do Agglomerative Clustering

Nesta etapa, implementamos o algoritmo Agglomerative Clustering com o número de clusters definido e treinamos o modelo nos dados de treino.

In [ ]:
# Treinamento do Agglomerative Clustering
n_clusters = 2  # ajuste conforme análise do dendrograma
agglo = AgglomerativeClustering(n_clusters=n_clusters, affinity='euclidean', linkage='ward')

# Ajustar nos dados de treino
train_labels = agglo.fit_predict(X_train)

# Como o clustering não usa os rótulos reais, é necessário mapear os clusters para as classes reais
def map_clusters_to_labels(cluster_labels, true_labels):
    from scipy.stats import mode
    labels = np.zeros_like(cluster_labels)
    for i in np.unique(cluster_labels):
        mask = cluster_labels == i
        labels[mask] = mode(true_labels[mask])[0]
    return labels

mapped_train_labels = map_clusters_to_labels(train_labels, y_train.values)

print("Rótulos mapeados para o conjunto de treino:")
print(pd.Series(mapped_train_labels).value_counts())

## 6. Avaliação do Modelo

Vamos avaliar o desempenho do modelo Agglomerative Clustering no conjunto de teste, utilizando métricas como acurácia e matriz de confusão.

In [ ]:
# Avaliação do modelo no conjunto de teste
# Prever clusters para o conjunto de teste
test_labels = agglo.fit_predict(X_test)
mapped_test_labels = map_clusters_to_labels(test_labels, y_test.values)

# Métricas de avaliação
acc = accuracy_score(y_test, mapped_test_labels)
cm = confusion_matrix(y_test, mapped_test_labels)
report = classification_report(y_test, mapped_test_labels)

print(f"Acurácia: {acc:.4f}")
print("Matriz de Confusão:")
print(cm)
print("\nRelatório de Classificação:")
print(report)

## 7. Repetição do Experimento

Para garantir a robustez dos resultados, o experimento pode ser repetido várias vezes com diferentes seeds e médias das métricas podem ser calculadas.

In [ ]:
# Repetição do experimento para robustez
n_runs = 5
accs = []

for seed in range(n_runs):
    X_train, X_test, y_train, y_test = train_test_split(
        X_norm, y, test_size=0.3, random_state=seed, stratify=y)
    agglo = AgglomerativeClustering(n_clusters=n_clusters, affinity='euclidean', linkage='ward')
    test_labels = agglo.fit_predict(X_test)
    mapped_test_labels = map_clusters_to_labels(test_labels, y_test.values)
    acc = accuracy_score(y_test, mapped_test_labels)
    accs.append(acc)

print(f"Acurácia média após {n_runs} execuções: {np.mean(accs):.4f} ± {np.std(accs):.4f}")

## 8. Análise dos Resultados e Considerações Finais

Nesta etapa, analisamos os resultados obtidos, discutindo o desempenho do Agglomerative Clustering no dataset Adult, possíveis limitações e sugestões para experimentos futuros.